# Практика: SFT на инструктивном сете
В предыдущем уроке вы познакомились с методами оптимизации обучения и реализовали некоторые из них, а в этом уроке обучите небольшую LLM на большем количестве данных. В процессе обучения вы будете измерять потребляемые ресурсы и сравнивать результаты с другими способами оптимизации.

Для обучения LLM вы примените библиотеку `TRL`, для дообучения моделей с адаптерами — `PEFT`, для уменьшения точности вычислений — `BitsAndBytes`, для логирования — `ClearML`. 

Задания выполняйте на ВМ в Jupyter Notebook.

## Подготовка данных
Возьмём часть SFT-датасета и приведём к формату, с которым удобно работать в `TRL`. Для примера возьмём первые 1000 строк из 150 тысяч из датасета [Vikhrmodels/GrandMaster-PRO-MAX](https://huggingface.co/datasets/Vikhrmodels/GrandMaster-PRO-MAX) с инструкциями и ответами в столбце `conversations`.

Для тренировки важно правильно подготовить формат данных. `TRL` ожидает определённую структуру сообщений, чтобы автоматически применить чат-шаблон модели.

### Задание 1

Напишите функцию, которая загрузит часть датасета и подготовит его для обучения с помощью `SFTTrainer`. Для этого:
Загрузите данные с помощью функции `load_dataset`.

В документации найдите, как должен выглядеть датасет для обучения через `TRL`, и подготовьте свои данные.

In [7]:
from datasets import load_dataset
from torch.utils.data import Subset

def load_first_k_examples(k=1000):
    # Загрузите датасет
    # Выберите нужное количество примеров (в зависимости от параметра k)
    # Преобразуйте формат, если необходимо (переименуйте нужный столбец, см. документацию)
    # Верните подготовленный датасет
    base_ds = load_dataset("Vikhrmodels/GrandMaster-PRO-MAX", split="train")
    return base_ds.select(range(k)).rename_column("conversation", "messages")

ds = load_first_k_examples()
ds

Dataset({
    features: ['source', 'messages', 'prompt_tokens', 'answer_tokens', 'cluster', 'prompt_lang', 'answer_lang'],
    num_rows: 1000
})

Теперь можно подать датасет в `SFTTrainer` и проверить, обработает ли он этот датасет.

### Обучение SFT
Попробуем запустить полноценный SFT без адаптеров.

### Задание 2
В коде зафиксированы гиперпараметры, чтобы логировать и фиксировать используемую память.

Добавьте другие гиперпараметры для обучения. Их значения можете выбрать на своё усмотрение. Рассмотрите параметры:
- `num_train_epochs`,
- `lr_scheduler_type`,
- `learning_rate`.

In [10]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from trl import SFTTrainer, SFTConfig
from transformers import TrainerCallback, TrainerState, TrainerControl

def run_classic_sft(ds):
    model_id = "Qwen/Qwen3-0.6B"
    tok = AutoTokenizer.from_pretrained(model_id)
    model = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype="auto")
    cfg = SFTConfig(
        output_dir="full_sft",
        per_device_train_batch_size=1,  
        logging_steps=1,
        max_length=1024,
        report_to='none',
        run_name='SFT',
        num_train_epochs=1,
        lr_scheduler_type="cosine",
        learning_rate=1e-4,
    )
    trainer = SFTTrainer(
        model=model,
        args=cfg,
        train_dataset=ds,
        processing_class=tok,
    )
    trainer.train()

ds = load_first_k_examples(100)
run_classic_sft(ds)

Truncating train dataset: 100%|██████████| 100/100 [00:00<00:00, 15045.75 examples/s]
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss
1,1.485742
2,1.797900
3,1.707693
4,2.239857
5,2.505584
6,2.454127
7,1.192542
8,2.219749
9,1.841383
10,1.839843


Writing model shards: 100%|██████████| 1/1 [00:01<00:00,  1.92s/it]


### Задание 3
Добавьте в код из предыдущего задания настройку параметров `LoRA` так, чтобы дообучались только матрицы проекций на q, k, v. Остальные гиперпараметры можете подобрать исходя из значений, которые вы видели в предыдущих уроках. Сравнение характеристик модели при разных параметрах приведено в [статье](https://arxiv.org/abs/2106.09685) (например, в таблице 6 — результаты тюнинга LoRA при разных r).

In [ ]:
from peft import LoraConfig

def run_lora_sft(ds):
    model_id = "Qwen/Qwen3-1.7B"
    tok = AutoTokenizer.from_pretrained(model_id)
    peft_cfg = LoraConfig(
        r=8, # ранг
        lora_alpha=16, # вес добавления адаптера
        lora_dropout=0.05,
        target_modules=["q_proj", "k_proj", "v_proj"],  # слои, к которым применяем 
    )
    cfg = SFTConfig(
        output_dir="sft-lora",
        per_device_train_batch_size=1,  
        logging_steps=1,
        max_length=1024,
        num_train_epochs=1,
        lr_scheduler_type="cosine",
        learning_rate=5e-5,
        report_to='none',
        run_name='LoRA'
    )
    model = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype="auto")
    trainer = SFTTrainer(
        model=model,
        args=cfg,
        train_dataset=ds,
        processing_class=tok,
        peft_config=peft_cfg,
    )
    trainer.train()
    
ds = load_first_k_examples(100)
run_lora_sft(ds)

Truncating train dataset: 100%|██████████| 100/100 [00:00<00:00, 4601.64 examples/s]
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss
1,1.513551
2,1.376103
3,1.761274
4,1.617273
5,2.963847
6,1.953071
7,1.200200
8,1.490595
9,1.693948
10,1.428861


Если на запуске обычного SFT у вас должно было затрачиваться порядка 9 ГБ видеопамяти, то на запуске SFT с LoRA этот показатель падает до 5 ГБ. Чтобы это проверить, запустите в терминале nvidia-smi. 

Поскольку параметров стало меньше, можно увеличить learning rate для скорости сходимости.

## SFT с int8
Попробуем сжать память весов самой модели с помощью библиотеки `BitsAndBytes`. 

Библиотека `BitsAndBytes` обеспечивает эффективное четырёх- и восьмибитное квантование для нейросетей. Она значительно снижает требования к видеопамяти при инференсе и обучении LLM: поддерживает восьмибитные оптимизаторы, умножение матриц и методы `QLoRA`.  
Согласно документации, все веса в 8 битах «как есть» не тренируют, поэтому мы будем тренировать LoRA-слои поверх квантизованной базы (это и есть классический QLoRA-подход). 

Посмотрим, насколько квантизация модели в 8-битную точность сократит потребляемую память в сравнении с предыдущим подходом.

### Задание 4
Добавьте в предыдущий код:
- инициализацию модели в 8 бит;
- оптимизатор в низкой точности.

Для этого создайте `BitsAndBytesConfig` с полем `load_in_8bit=True` и передайте его в поле `quantization_config` при создании модели с помощью `AutoModelForCausalLM.from_pretrained(…)`.

In [ ]:
from transformers import BitsAndBytesConfig

def run_int8_sft(ds):
    model_id = "Qwen/Qwen3-0.6B"
    tok = AutoTokenizer.from_pretrained(model_id)
    peft_cfg = LoraConfig(
        r=8, # ранг
        lora_alpha=16, # вес добавления адаптера
        lora_dropout=0.05,
        target_modules=["q_proj", "k_proj", "v_proj"],  # слои, к которым применяем 
    )
    cfg = SFTConfig(
        output_dir="sft-int8",
        per_device_train_batch_size=1,  
        logging_steps=1,
        max_length=1024,
        num_train_epochs=1,
        lr_scheduler_type="cosine",
        learning_rate=5e-5,
        report_to='none',
        run_name='8bit',
        optim="adamw_bnb_8bit",
    )
    quantization_config = BitsAndBytesConfig(
        load_in_8bit=True,
    )
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        torch_dtype="auto",
        quantization_config=quantization_config,
    )
    trainer = SFTTrainer(
        model=model, 
        args=cfg, 
        train_dataset=ds,
        processing_class=tok,
        peft_config=peft_cfg,
    )
    trainer.train()

ds = load_first_k_examples()
run_int8_sft(ds)